# 08 — Interpretabilidade do XGBoost

Extrai feature importance do XGBoost e agrega as features derivadas do One-Hot Encoding para recuperar a visão por variável original.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import joblib

PASTA_PROJETO = "TCC_PRF"
PASTA_MODELOS = os.path.join(PASTA_PROJETO, "outputs", "modelos")
PASTA_INTERPRETACAO = os.path.join(PASTA_PROJETO, "outputs", "interpretabilidade")
os.makedirs(PASTA_INTERPRETACAO, exist_ok=True)

pipeline = joblib.load(os.path.join(PASTA_MODELOS, "xgboost.joblib"))
preprocessador = pipeline.named_steps["preprocessamento"]
modelo_xgb = pipeline.named_steps["modelo"]

nomes_features = preprocessador.get_feature_names_out()
importancias = modelo_xgb.feature_importances_

feature_importance = (pd.DataFrame({"feature": nomes_features, "importance": importancias})
                      .sort_values("importance", ascending=False)
                      .reset_index(drop=True))
display(feature_importance.head(30))

In [ ]:
feature_importance.to_csv(
    os.path.join(PASTA_INTERPRETACAO, "feature_importance_xgboost.csv"),
    index=False, encoding="utf-8-sig"
)

top25 = feature_importance.head(25).sort_values("importance")
plt.figure(figsize=(11,9))
plt.barh(top25["feature"], top25["importance"])
plt.title("25 variáveis mais importantes - XGBoost")
plt.xlabel("Importância")
plt.tight_layout()
plt.savefig(os.path.join(PASTA_INTERPRETACAO, "top25_feature_importance.png"),
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
categoricas = [
    "uf","br_categoria","dia_semana","fase_dia","sentido_via",
    "condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"
]

def identificar_variavel_original(feature):
    nome = feature.replace("categoricas__", "").replace("numericas__", "")
    for variavel in sorted(categoricas, key=len, reverse=True):
        if nome == variavel or nome.startswith(variavel + "_"):
            return variavel
    return nome

feature_importance["variavel_original"] = feature_importance["feature"].apply(identificar_variavel_original)

importancia_agrupada = (
    feature_importance.groupby("variavel_original", as_index=False)
    .agg(importance=("importance","sum"))
    .sort_values("importance", ascending=False)
)
importancia_agrupada["percentual_importancia"] = (
    importancia_agrupada["importance"] / importancia_agrupada["importance"].sum() * 100
)
display(importancia_agrupada)

importancia_agrupada.to_csv(
    os.path.join(PASTA_INTERPRETACAO, "importancia_variaveis_agrupadas.csv"),
    index=False, encoding="utf-8-sig"
)

In [ ]:
config_modelo_final = pd.DataFrame({
    "parametro": ["modelo","threshold","criterio_threshold","periodo_treino","periodo_validacao","periodo_teste_final"],
    "valor": ["XGBoost","0.28","Maior F2-score","2020-2023","2024","2025"]
})
config_modelo_final.to_csv(
    os.path.join(PASTA_MODELOS, "configuracao_modelo_final.csv"),
    index=False, encoding="utf-8-sig"
)
display(config_modelo_final)
print("2025 ainda não foi utilizado.")